In [0]:
from pyspark.sql.functions import * 
from delta.tables import * 
from pyspark.sql.window import *

In [0]:
%run /Workspace/Users/narutosha289@gmail.com/Consolidated_pipeline/Setup/Utilities 

In [0]:
dbutils.widgets.text("Catalog","FMCG","Catalog") 
dbutils.widgets.text("Data_Source","gross_price","Data_Source") 

Catalog = dbutils.widgets.get("Catalog")
Data_Source = dbutils.widgets.get("Data_Source")
BasePath = "s3a://sportsbar-us1/gross_price/gross_price.csv"
print(BasePath)

s3a://sportsbar-us1/gross_price/gross_price.csv


### Bronze Transformation

In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(BasePath)
        .withColumn("read_timestamp", current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
display(df.limit(10))

product_id,month,gross_price,read_timestamp,file_name,file_size
25891101,2025/07/01,-84,2026-09-11T15:36:23.823Z,gross_price.csv,2741
25891101,01/08/2025,unknown,2026-09-11T15:36:23.823Z,gross_price.csv,2741
25891101,2025/09/01,84,2026-09-11T15:36:23.823Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-09-11T15:36:23.823Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-09-11T15:36:23.823Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-09-11T15:36:23.823Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-09-11T15:36:23.823Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-09-11T15:36:23.823Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-09-11T15:36:23.823Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-09-11T15:36:23.823Z,gross_price.csv,2741


In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{Catalog}.{bronze_schema}.{Data_Source}")

### Silver Transformations

In [0]:
df_bronze = spark.sql(f"select * from {Catalog}.{bronze_schema}.{Data_Source}") 
df_bronze.show()

+----------+----------+-------------+--------------------+---------------+---------+--------+--------------------+-----------+---------+
|product_id|     month|  gross_price|      read_timestamp|      file_name|file_size|order_id|order_placement_date|customer_id|order_qty|
+----------+----------+-------------+--------------------+---------------+---------+--------+--------------------+-----------+---------+
|  25891101|2025/07/01|          -84|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|01/08/2025|      unknown|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|2025/09/01|           84|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|2025-10-01|           83|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|2025-11-01|           83|2026

In [0]:
df_bronze.select("month").distinct().show()

+----------+
|     month|
+----------+
|2025/07/01|
|01/08/2025|
|2025/09/01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025/11/01|
|2025/08/01|
|01-09-2025|
|2025/10/01|
|01/12/2025|
|01/09/2025|
|01-12-2025|
|01-08-2025|
|01/10/2025|
+----------+



In [0]:
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"] 

df_silver = df_bronze.withColumn("month", coalesce(*[try_to_date(df_bronze.month, fmt) for fmt in date_formats]))

In [0]:
df_silver.show()

+----------+----------+-------------+--------------------+---------------+---------+--------+--------------------+-----------+---------+
|product_id|     month|  gross_price|      read_timestamp|      file_name|file_size|order_id|order_placement_date|customer_id|order_qty|
+----------+----------+-------------+--------------------+---------------+---------+--------+--------------------+-----------+---------+
|  25891101|2025-07-01|          -84|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|2025-08-01|      unknown|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|2025-09-01|           84|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|2025-10-01|           83|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|2025-11-01|           83|2026

In [0]:
df_silver = df_silver.withColumn("gross_price", 
    when(df_silver["gross_price"].rlike(r'^-?\d+(\.\d+)?$'), 
        when(df_silver["gross_price"].cast("double") < 0, -1*df_silver["gross_price"].cast("double"))
        .otherwise(df_silver["gross_price"].cast("double"))
    ).otherwise(0)
)

In [0]:
df_silver.show()

+----------+----------+-----------+--------------------+---------------+---------+--------+--------------------+-----------+---------+
|product_id|     month|gross_price|      read_timestamp|      file_name|file_size|order_id|order_placement_date|customer_id|order_qty|
+----------+----------+-----------+--------------------+---------------+---------+--------+--------------------+-----------+---------+
|  25891101|2025-07-01|       84.0|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|2025-08-01|        0.0|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|2025-09-01|       84.0|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|2025-10-01|       83.0|2026-09-11 15:36:...|gross_price.csv|     2741|    NULL|                NULL|       NULL|     NULL|
|  25891101|2025-11-01|       83.0|2026-09-11 15:36:...

#### We join the pricing table with products table to get product code for the items! 

In [0]:
df_products = spark.table("fmcg.silver.products").alias("p")
df_silver_aliased = df_silver.alias("s")

df_joined = df_silver_aliased.join(
    df_products,
    col("s.product_id") == col("p.product_id"),
    "inner"
)

df_joined = df_joined.select(
    "s.product_id", "p.product_code", "s.month", "s.gross_price",
    "s.read_timestamp", "s.file_name", "s.file_size"
)
df_silver_aliased.printSchema()
df_joined.printSchema()
spark.table(f"{Catalog}.{silver_schema}.{Data_Source}").printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- month: date (nullable = true)
 |-- gross_price: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_qty: double (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_code: string (nullable = true)
 |-- month: date (nullable = true)
 |-- gross_price: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_code: string (nullable = true)
 |-- month: date (nullable = true)
 |-- gross_price: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- fil

In [0]:
df_joined.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{Catalog}.{silver_schema}.{Data_Source}")

### Gold Processing

In [0]:
df_silver = spark.sql(f"SELECT * FROM {Catalog}.{silver_schema}.{Data_Source};")
df_gold = df_silver.select("product_code", "month", "gross_price")
df_gold.show(5)

+--------------------+----------+-----------+
|        product_code|     month|gross_price|
+--------------------+----------+-----------+
|e91ba9d665f90254d...|2025-07-01|       84.0|
|e91ba9d665f90254d...|2025-08-01|        0.0|
|e91ba9d665f90254d...|2025-09-01|       84.0|
|e91ba9d665f90254d...|2025-10-01|       83.0|
|e91ba9d665f90254d...|2025-11-01|       83.0|
+--------------------+----------+-----------+
only showing top 5 rows


In [0]:
df_gold.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{Catalog}.{gold_schema}.sb_dim_{Data_Source}")

### Merging data source with parent

In [0]:
df_gold_price = spark.table("fmcg.gold.sb_dim_gross_price")
df_gold_price.show(5)

+--------------------+----------+-----------+
|        product_code|     month|gross_price|
+--------------------+----------+-----------+
|e91ba9d665f90254d...|2025-07-01|       84.0|
|e91ba9d665f90254d...|2025-08-01|        0.0|
|e91ba9d665f90254d...|2025-09-01|       84.0|
|e91ba9d665f90254d...|2025-10-01|       83.0|
|e91ba9d665f90254d...|2025-11-01|       83.0|
+--------------------+----------+-----------+
only showing top 5 rows


### Get the price for each product code(aggeregated by year)

In [0]:
df_gold_price = (
    df_gold_price
    .withColumn("year", year("month"))
    # 0 = non-zero price, 1 = zero price  ➜ non-zero comes first
    .withColumn("is_zero", when(col("gross_price") == 0, 1).otherwise(0))
)

w = (
    Window
    .partitionBy("product_code", "year")
    .orderBy(col("is_zero"), col("month").desc())
)


df_gold_latest_price = (
    df_gold_price
      .withColumn("rnk", row_number().over(w))
      .filter(col("rnk") == 1)
)


df_gold_latest_price = df_gold_latest_price.select("product_code", "year", "gross_price").withColumnRenamed("gross_price", "price_inr").select("product_code", "price_inr", "year")

# change year to string
df_gold_latest_price = df_gold_latest_price.withColumn("year", col("year").cast("string"))

df_gold_latest_price.show(5)

+--------------------+---------+----+
|        product_code|price_inr|year|
+--------------------+---------+----+
|062f5574bbdf4386b...|    281.0|2025|
|0cb7b2f42657b625f...|    100.0|2025|
|102628255d24304d6...|     86.0|2025|
|2e387cef1424d6e7b...|    108.0|2025|
|3cab59f0592428527...|    493.0|2025|
+--------------------+---------+----+
only showing top 5 rows


In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_gross_price")

delta_table.alias("target").merge(
    source=df_gold_latest_price.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]